In [ ]:
import pandas as pd
import numpy as np
import loaders
import gensim.models

In [ ]:
tisch = loaders.load_tisch()
sept = loaders.load_sept()

### Creating Strongs Word Vectors

##### Creating strong's df

In [ ]:
tisch_str_verses = (
    tisch.groupby(["book", "chapter", "verse"])["str"].agg(lambda x: " ".join(x)).values
)
sept_str_verses = (
    sept.groupby(["book", "chapter", "verse"])["str"].agg(lambda x: " ".join(x)).values
)

display(tisch_str_verses[:3])

In [ ]:
bible_strs = np.concatenate([tisch_str_verses, sept_str_verses])
print("actual len:", len(tisch_str_verses) + len(sept_str_verses))
print("hopeful len:", len(bible_strs))

In [ ]:
try:
    model = gensim.models.Word2Vec.load("word2vec_strs.model")
except FileNotFoundError:
    model = gensim.models.Word2Vec(vector_size=100, window=5, min_count=1, workers=4)
    model.build_vocab(bible_strs)
    model.train(
        bible_strs,
        total_examples=model.corpus_count,
        epochs=100,
        compute_loss=True,
    )
    model.save("word2vec.model")